Textual Cleaning of ECB and FED Stability Reports and MOnetary policy reports, effectively all teh main tet that I will not be labelling. 

In [63]:
# Importing the operating system interface, basically where I will interact with the file system. 
import os
# Used when I need to asscertain pattern matching and or text manipulation.
import re
# Provides access to command-line arguments
import sys
# The actual format the OLMO model will be able to read, Since everything that is getting fed through is acctually a Txt file. 
import json
# Filename pattern matching
import glob
# Command-line argument parser 
import argparse
# Cryptographic hashing
import hashlib
# Unicode character database
import unicodedata
# Fixes Text For You
import ftfy
# Encodes text into tokens
import tiktoken
# Path package
from pathlib import Path
# only needed for the Fred Speeches Conversion
import pandas as pd

File paths where my documents lie, alongisde configuration for my model and tokeniser

In [64]:
data_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report")
INPUT_DIRS = [data_path]    
OUT_PATH = "FED_annualreport_unlablled.jsonl"

# Configuration so that it is inline with the OLMO 2 modles
# Effetcively the max numbers of tokens that the model can process, for anything over it is handeled by splitting. 
TARGET_TOKENS = 2048
# For text that isn't relevant and ost likely boiler plates from a failed scrape
MIN_TOKENS = 64
# discards chunks shorter than this many characters
MIN_CHARS = 200
# discards chunks with a lower share of letters (tables / number dumps)
MIN_ALPHA = 0.60
# drop standalone 'Sources:/Notes:/Chart' furniture lines
DROP_NOTES = True

Extra Step Need for The FOMC minutes

In [65]:
#data = pd.read_csv(data_path / "ecb_speeches.csv")
#data.head(10)
#all_text = "\n".join(data["text"].dropna().astype(str))
#with open("fed_FOMC_remaining_sentences.txt", "w", encoding="utf-8") as f:
    #f.write(all_text)


Extra Step for Fed Speeches since they need to be first converted into txt files before running.

In [66]:
#data = pd.read_csv(data_path / "fed_speeches.csv")
#all_text = "\n".join(data["text"].dropna().astype(str))
#with open("fed_speeches_merged.txt", "w", encoding="utf-8") as f:
    #f.write(all_text)

Tokeniser

In [67]:
# The justifcation for picking this is since the cl100k_base
# shares the base vocab, the token counts will match that of the dolma2 tokenizer
enc = tiktoken.get_encoding("cl100k_base")
def n_tokens(s): return len(enc.encode(s))


Boiler Plate Cleaning

In [68]:
"""
Cleaning pipeline for scraped Federal Reserve documents (Monetary Policy
Reports, Financial Stability Reports, FOMC material, ...) ahead of continued
pre-training of an LLM.

Drop-in replacement for the previous clean_text(): same name, same
drop_notes default, plus a few optional knobs.

The stages run in this order, and the order matters:

  1. Unicode / mojibake repair (ftfy -> fraction handling -> NFKC -> tidy).
  2. Line-level junk removal: running headers/footers, transmittal
     boilerplate, chart/table debris, NOTE/SOURCE/footnote blocks.
     This happens BEFORE de-hyphenation because a running header often sits
     *between* the two halves of a word hyphenated across a page break
     ("communi-\\nBoard of Governors...\\ncation") and must be gone before
     the halves can be re-joined.
  3. De-hyphenation across line breaks ("communi-\\ncation" -> "communication").
  4. Paragraph reflow: unwrap the hard line breaks PDF extraction leaves
     every ~45-90 characters so the model trains on natural paragraphs
     instead of arbitrarily chopped lines. Blank lines (paragraph
     boundaries) are preserved through the whole pipeline for this reason.
  5. Punctuation / whitespace tidy-up.

Corpus-level steps that belong OUTSIDE this function: exact/near-duplicate
deduplication across documents (the same MPR often exists as both HTML and
PDF scrapes), dropping documents that come out shorter than some minimum
after cleaning, and manually spot-checking a random sample of outputs.
"""

import re
import unicodedata
from collections import Counter

# ftfy is strongly recommended (fixes mojibake like "â€™" -> "'"), but the
# pipeline degrades gracefully if it is not installed.
try:
    import ftfy
    _HAS_FTFY = True
except ImportError:
    _HAS_FTFY = False


# ---------------------------------------------------------------------------
# Stage 1 -- Unicode normalization
# ---------------------------------------------------------------------------

# Precomposed vulgar fractions are mapped BEFORE NFKC. NFKC would turn
# "5\u00bc" into "51\u20444" (superscript/subscript digits flattened and glued
# onto the preceding integer), which is exactly the ambiguity we want to avoid.
_VULGAR_FRACTIONS = {
    "\u00bc": " 1/4", "\u00bd": " 1/2", "\u00be": " 3/4",
    "\u2153": " 1/3", "\u2154": " 2/3",
    "\u2155": " 1/5", "\u2156": " 2/5", "\u2157": " 3/5", "\u2158": " 4/5",
    "\u2159": " 1/6", "\u215a": " 5/6",
    "\u215b": " 1/8", "\u215c": " 3/8", "\u215d": " 5/8", "\u215e": " 7/8",
}

_SUPERSCRIPT_DIGITS = "\u2070\u00b9\u00b2\u00b3\u2074\u2075\u2076\u2077\u2078\u2079"
_SUBSCRIPT_DIGITS = "\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089"
_SUP_TO_ASCII = str.maketrans(_SUPERSCRIPT_DIGITS, "0123456789")
_SUB_TO_ASCII = str.maketrans(_SUBSCRIPT_DIGITS, "0123456789")

# "5{sup}1{frac-slash}{sub}4" -> "5 1/4"; runs before NFKC for the same reason.
_RE_SUPSUB_FRACTION = re.compile(
    "([%s]+)\\s*\u2044\\s*([%s]+)" % (_SUPERSCRIPT_DIGITS, _SUBSCRIPT_DIGITS)
)

# Fractions already glued by an earlier NFKC pass in the scraper:
# "51\u20444 percent" -> "5 1/4 percent". This only fires on U+2044 FRACTION
# SLASH -- never on the ordinary "/" -- so dates like 1/2/2004 are untouched.
# Single-digit numerators only ("5 11\u204416" stays ambiguous and is left alone).
_RE_GLUED_FRACTION = re.compile("(\\d+)([1-9])\u2044(\\d{1,2})")

# Soft hyphen used as a discretionary line-break hyphen.
_RE_SOFT_HYPHEN_JOIN = re.compile("(\\w)\u00ad\\s*\\n?\\s*(\\w)")

# Hyphen look-alikes normalized to ASCII "-". En/em dashes are real
# punctuation and are kept.
_RE_HYPHEN_VARIANTS = re.compile("[\u2010\u2011\u2012\u2212]")

# Bullets, middle dots, and the black squares used in MPR running headers
# ("Monetary Policy Report \u25aa February 2019").
_RE_BULLETS = re.compile("[\u00b7\u2022\u2023\u25aa\u25a0\u25cf\u25b6\u25b8\u2043]+")

# Zero-widths, BOM, and the replacement character.
_RE_INVISIBLES = re.compile("[\u200b-\u200f\ufeff\ufffd]")

# C0 control characters other than \n and \t.
_RE_CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")


def _normalize_unicode(text):
    if _HAS_FTFY:
        text = ftfy.fix_text(text)
    text = text.replace("\u2028", "\n").replace("\u2029", "\n\n")
    text = _RE_SUPSUB_FRACTION.sub(
        lambda m: " %s/%s" % (m.group(1).translate(_SUP_TO_ASCII),
                              m.group(2).translate(_SUB_TO_ASCII)),
        text,
    )
    for char, replacement in _VULGAR_FRACTIONS.items():
        text = text.replace(char, replacement)
    text = unicodedata.normalize("NFKC", text)
    text = _RE_GLUED_FRACTION.sub(r"\1 \2/\3", text)
    text = text.replace("\u2044", "/")
    text = text.replace("\u00a0", " ")          # NBSP (NFKC already handles it; belt and braces)
    text = _RE_INVISIBLES.sub("", text)
    text = _RE_SOFT_HYPHEN_JOIN.sub(r"\1\2", text)
    text = text.replace("\u00ad", "")
    text = _RE_HYPHEN_VARIANTS.sub("-", text)
    text = _RE_BULLETS.sub(" ", text)
    text = _RE_CONTROL.sub(" ", text)
    return text


# ---------------------------------------------------------------------------
# Stage 2 -- Line-level boilerplate / chart-debris removal
# ---------------------------------------------------------------------------
# Every pattern here is applied with fullmatch() against a single stripped
# line. This is the crucial difference from the old code, where unanchored
# patterns such as "February \\d{1,2}, \\d{4}", "Wednesday", "Contents", and
# "Figure\\s+\\d+" (all with IGNORECASE) deleted words out of the middle of
# body sentences and left broken text behind. Full-line matching means an
# in-sentence "On February 14, 2023, the Committee..." survives, while a
# standalone dateline on the cover page is dropped.

_MONTHS = (r"(?:january|february|march|april|may|june|july|august|"
           r"september|october|november|december)")

# Case-insensitive full-line boilerplate.
_CI_LINE_PATTERNS = [
    # Cover page / transmittal letter furniture
    r"for (?:use|release) at \d{1,2}:\d{2}\s*[ap]\.?\s?m\.?,?\s*e?[cdmps]*\.?[ds]?\.?t\.?\,?.*",
    r"embargoed for release.*",
    r"board of governors of the federal reserve system(?:\s*[|,].*)?",
    r"monetary policy report(?: to the congress)?(?:\s*[|,]?\s*" + _MONTHS + r"\s+\d{4})?",
    r"financial stability report(?:\s*[|,]?\s*" + _MONTHS + r"\s+\d{4})?",
    r"letter of transmittal",
    r"the president of the senate",
    r"the speaker of the house(?: of representatives)?",
    r"dear (?:mr|madam|ms|mrs)\.?\s+(?:president|speaker)\s*[,.:;]?",
    r"(?:submitted )?pursuant to section 2b of the federal reserve act.*",
    # Any chair's signature line, not just Greenspan's -- the corpus spans decades.
    r"[a-z .,'-]{3,40},\s*chair(?:man|woman)?(?:\s+pro tempore)?",
    r"washington,?\s*d\.?\s?c\.?,?(?:\s+" + _MONTHS + r"\s+\d{1,2},?\s+\d{4})?\.?",
    r"sincerely,?",
    r"contents",
    r"list of (?:boxes|figures|tables)",
    r"(?:report )?submitted to the congress on\s+.*",
    r"page(?:\s+\d+)?",
    # Website chrome, in case some documents were scraped from federalreserve.gov HTML
    r"skip to main content",
    r"back to top",
    r"last update:?\s*.*",
    r"accessible version(?:\s+of.*)?",
    r"return to text",
    r"printable version",
    r"pdf(?:\s*\|\s*html)?",
    r"https?://\S+",
    r"www\.\S+",
    # Standalone datelines / weekday lines (full line only -> in-sentence dates survive)
    r"(?:mon|tues|wednes|thurs|fri|satur|sun)day(?:,\s+" + _MONTHS + r"\s+\d{1,2},?\s+\d{4})?",
    _MONTHS + r"\s+\d{1,2},?\s+\d{4}",
    # Chart axis / unit lines
    r"percent(?:age points)?(?:,\s*annual rate)?",
    r"percent,\s*(?:monthly|quarterly|weekly|daily|annual rate)",
    r"(?:billions?|trillions?|millions?|thousands?) of (?:\d{4} )?(?:chained\s*\(\d{4}\)\s*)?dollars",
    r"dollars per \w+",
    r"basis points",
    r"index(?:,.*=\s*100.*)?",
    r".{0,30}=\s*100",
    r"ratio(?:\s+scale)?",
    r"log scale.*",
    r"\d{1,2}-month percent change",
    r"annual(?:ized)? rate",
    r"seasonally adjusted(?:\s+annual rate)?",
    r"(?:quarterly|monthly|weekly|daily)(?:\s+average)?",
    r"panel\s+[a-z]",
    r"(?:\()?continued(?:\))?",
    r"h\.\d{1,2}(?:\.\d+)?(?:\s+release)?",   # statistical release codes, e.g. H.15
]
_CI_LINE = re.compile(
    "(?:%s)" % "|".join("(?:%s)" % p for p in _CI_LINE_PATTERNS), re.IGNORECASE
)

# Case-SENSITIVE full-line patterns. Kept case-sensitive on purpose: a body
# sentence wrapped mid-line starts in lowercase ("...as shown in\nfigure 2,
# rates fell"), so requiring the capitalized caption form plus punctuation
# ("Figure 2. Change in real GDP") avoids eating in-text cross-references.
_CS_LINE_PATTERNS = [
    r"(?:Figure|Chart|Table|Box|Exhibit)\s+[A-D]?\.?\d+(?:\.\d+)?[.:]\s*\S.*",
    r"(?:Figure|Chart|Table|Box|Exhibit)\s+[A-D]?\.?\d+(?:\.\d+)?",
    r"[A-Z][a-z]+\s+[A-Z][a-z]+\s+\d+",       # axis tick rows like "January February 2004"
    # The curated chart-title list (built with LLM help from extracts of every
    # report year). Kept, but now full-line + case-sensitive so a wrapped body
    # line beginning "net percentage of domestic banks tightening..." survives.
    r"Selected interest rates",
    r"Ten-year Treasury",
    r"Two-year Treasury",
    r"Intended federal funds rate",
    r"Change in real GDP",
    r"Change in PCE chain-type price index",
    r"Change in real income and consumption",
    r"Wealth-to-income ratio",
    r"Change in house prices",
    r"Mortgage rates",
    r"Private housing starts",
    r"Household financial obligations ratio",
    r"Delinquency rates on selected types of household loans",
    r"Change in real business fixed investment",
    r"Change in real business inventories",
    r"Before-tax profits of nonfinancial corporations.*",
    r"Selected components of net business financing.*",
    r"Financing gap and net equity retirement.*",
    r"Net percentage of domestic banks tightening.*",
    r"Default rate on outstanding corporate bonds",
    r"Federal receipts and expenditures",
    r"Net saving",
    r"Change in real government expenditures.*",
    r"Federal government debt held by the public",
    r"Treasury securities held by foreign investors.*",
    r"State and local government net saving",
    r"U\.S\. trade and current account balances",
    r"Prices of oil and of nonfuel commodities",
    r"Prices of major nonfuel commodities",
    r"U\.S\. net financial inflows",
    r"U\.S\. net international securities transactions",
    r"Civilian unemployment rate",
    r"Net change in payroll employment",
    r"Labor force participation rate",
    r"Measures of change in hourly compensation",
    r"Change in PCE prices excluding food and energy",
    r"Alternative measures of price change",
    r"Change in consumer prices.*",
    r"TIPS-based inflation compensation",
    r"Change in unit labor costs",
    r"Interest rates on selected Treasury securities",
    r"Spreads of corporate bond yields over.*",
    r"Implied S&P 500 volatility",
    r"Stock price indexes",
    r"Growth of domestic nonfinancial debt",
    r"M2 growth rate",
    r"Equity indexes in selected foreign industrial countries",
    r"Spread on internationally issued sovereign debt.*",
    r"Official interest rates in selected foreign industrial countries",
    r"U\.S\. dollar nominal exchange rate.*",
    r"U\.S\. dollar exchange rate against.*",
    r"Equity indexes in selected emerging-market economies",
]
_CS_LINE = re.compile("(?:%s)" % "|".join("(?:%s)" % p for p in _CS_LINE_PATTERNS))

# NOTE / SOURCE / numbered-footnote openers. When one of these is seen (and
# drop_notes=True), the line AND its wrapped continuation lines are skipped
# until the next blank line -- the old version only removed the first line of
# a note and left the rest behind.
_RE_NOTE_START = re.compile(r"(?:NOTE|NOTES|SOURCE|SOURCES|N\.B\.)\s*[:.\u2014-]", re.IGNORECASE)
# "12. See the minutes of the March meeting..." -- classic footnote body.
# Caveat: this would also remove a numbered list whose items start with a
# capitalized sentence; those are essentially absent from Fed reports, but
# flip drop_notes off if that ever matters for another corpus.
_RE_FOOTNOTE_START = re.compile(r"\d{1,2}\.\s+[A-Z\u201c\"(]")

# Chart-noise line shapes (generalizations of the old per-pattern checks).
_RE_NUMERIC_JUNK = re.compile(r"[\d\s()+\-*/=.:,%$\u2013\u2014]+")   # digits/punct only
_RE_QUARTER_TICKS = re.compile(r"(?:[QH][1-4]\s*)+", re.IGNORECASE)
_RE_MONTH_TICKS = re.compile(
    r"(?:(?:jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)[a-z]{0,6}\.?\s*)+",
    re.IGNORECASE,
)
_RE_SHORT_ACRONYM = re.compile(r"[A-Z]{1,6}\d{0,2}")                  # "GDP", "PCE", "M2" alone
_RE_ALLCAPS_LINE = re.compile(r"[A-Z][A-Z ,.:;&'()\d-]{7,}")          # running heads in caps

_RE_SENTENCE_END = re.compile("[.!?][)\\]\"'\u201d\u2019]*$")


def _is_junk_line(line):
    """Structural (non-lexical) chart/table debris checks for one stripped line."""
    if _RE_NUMERIC_JUNK.fullmatch(line):
        return True
    if _RE_QUARTER_TICKS.fullmatch(line) or _RE_MONTH_TICKS.fullmatch(line):
        return True
    if len(line) <= 8 and _RE_SHORT_ACRONYM.fullmatch(line):
        return True
    alpha = sum(1 for c in line if c.isalpha())
    if alpha < 3:
        return True
    # Low letter-to-character ratio => axis labels / data rows. Only applied
    # to ASCII lines so non-Latin text is never penalized.
    if line.isascii() and alpha / len(line) < 0.15:
        return True
    return False


def _filter_lines(text, drop_notes, drop_allcaps, min_repeat):
    lines = [line.strip() for line in text.split("\n")]

    # Frequency-based running header/footer removal. "Monetary Policy Report
    # [sq] July 2019" style headers repeat on every page; any short line that
    # repeats min_repeat+ times and does not end like a sentence is dropped.
    # This is what makes the cleaner year- and report-agnostic instead of
    # depending on the hardcoded lists alone.
    counts = Counter(line for line in lines if line)

    kept = []
    in_note_block = False
    prev_kept = ""
    for line in lines:
        if not line:
            in_note_block = False
            prev_kept = ""
            kept.append("")            # blank lines = paragraph boundaries; keep them
            continue
        if in_note_block:
            continue
        # Footnote bodies ("12. See the minutes...") are only treated as such
        # at a paragraph start or after a finished sentence -- otherwise a
        # body line that happens to wrap right before a small number
        # ("...met on May\n14. The Committee...") would be eaten.
        looks_like_footnote = (
            _RE_FOOTNOTE_START.match(line)
            and (not prev_kept or _RE_SENTENCE_END.search(prev_kept))
        )
        if drop_notes and (_RE_NOTE_START.match(line) or looks_like_footnote):
            in_note_block = True
            continue
        if (len(line) <= 90 and counts[line] >= min_repeat
                and not _RE_SENTENCE_END.search(line)):
            continue
        if _CI_LINE.fullmatch(line) or _CS_LINE.fullmatch(line):
            continue
        if drop_allcaps and " " in line and _RE_ALLCAPS_LINE.fullmatch(line):
            continue
        if _is_junk_line(line):
            continue
        kept.append(line)
        prev_kept = line
    return "\n".join(kept)


# ---------------------------------------------------------------------------
# Stage 3 -- De-hyphenation across line breaks
# ---------------------------------------------------------------------------
# Only newline-hyphenation is joined. The old rule (\w+)-\s+(\w+) also merged
# hyphen + ordinary space, which corrupted suspended hyphens that are
# everywhere in Fed prose: "short- and long-term rates" became
# "shortand long-term rates". Same-line hyphens are now left alone.

_RE_LINEBREAK_HYPHEN = re.compile(r"(\w+)-\n(\w+)")
_SUSPENDED_HYPHEN_WORDS = {"and", "or", "to", "the"}   # "short- and", "one- to three-year"


def _dehyphenate(match):
    left, right = match.group(1), match.group(2)
    if right.lower() in _SUSPENDED_HYPHEN_WORDS:
        return left + "- " + right                     # "short-\nand" -> "short- and"
    if right[0].isupper():
        return left + "-" + right                      # "non-\nOPEC" -> "non-OPEC"
    if right[0].isdigit():
        return left + "-" + right                      # "2-\n3 percent" -> "2-3 percent"
    return left + right                                # "communi-\ncation" -> "communication"


# ---------------------------------------------------------------------------
# Stage 4 -- Paragraph reflow
# ---------------------------------------------------------------------------

def _reflow(text, wrap_width=45):
    """Unwrap hard line breaks inside paragraphs.

    Heuristics for merging a line with the one after it:
      - next line starts with a lowercase letter (strongest signal), or
      - current line ends with a comma/semicolon, or
      - current line is at least wrap_width chars and does not end like a
        sentence (typical mid-paragraph wrap), including when the next line
        starts with a digit ("...rose\\n2.5 percent").
    Headings ("Economic and Financial Developments") are short and are
    followed by a capitalized line, so they keep their own line. wrap_width
    defaults low because MPR PDFs are two-column and extract at ~45-60
    chars per line.
    """
    merged = []
    current = ""
    for raw in text.split("\n"):
        line = raw.strip()
        if not line:
            if current:
                merged.append(current)
                current = ""
            merged.append("")
            continue
        if not current:
            current = line
            continue
        join = (
            line[0].islower()
            or line[0] in ",;)"
            or current.endswith((",", ";"))
            or (len(current) >= wrap_width and not _RE_SENTENCE_END.search(current))
        )
        if join:
            current = current + " " + line
        else:
            merged.append(current)
            current = line
    if current:
        merged.append(current)
    return "\n".join(merged)


# ---------------------------------------------------------------------------
# Stage 5 -- Final tidy-up
# ---------------------------------------------------------------------------

# In-text footnote markers left glued to sentences by PDF extraction:
# "...policy rates.12 The Committee" -> "...policy rates. The Committee".
# The fixed-width lookbehind requires letter-or-")" then punctuation, so
# decimals ("1.5 percent") and codes ("M2.") are never touched.
_RE_FOOTNOTE_MARKER = re.compile(r"(?<=[a-z)][.;:,])\d{1,2}(?=\s|\Z)")

_RE_SPACE_BEFORE_PUNCT = re.compile(r"[ \t]+([,.;:!?%])")
_RE_SPACE_AFTER_OPEN = re.compile(r"([(\[])\s+")
_RE_SPACE_BEFORE_CLOSE = re.compile(r"\s+([)\]])")


def clean_text(text, drop_notes=True, reflow=True, drop_allcaps=True,
               min_repeat=3, min_alpha_chars=0):
    """Clean one scraped Fed document for LLM continued pre-training.

    Parameters
    ----------
    drop_notes : remove NOTE./SOURCE. blocks, numbered footnote bodies
        (including their wrapped continuation lines), and in-text footnote
        markers ("rates.12" -> "rates.").
    reflow : unwrap hard line breaks so the model sees whole paragraphs.
    drop_allcaps : drop ALL-CAPS lines of two or more words (running heads
        in older reports). Set False to keep ALL-CAPS section headings.
    min_repeat : a short line repeated this many times in the document is
        treated as a running header/footer and removed.
    min_alpha_chars : if the cleaned text has fewer alphabetic characters
        than this, return "" so the caller can drop the document.
    """
    if not text:
        return ""

    # 1. Unicode
    text = _normalize_unicode(text)

    # 2. Line-level boilerplate / chart debris (before de-hyphenation, so a
    #    running header between two halves of a hyphenated word is gone
    #    before the join is attempted).
    text = _filter_lines(text, drop_notes, drop_allcaps, min_repeat)

    # 3. De-hyphenation across line breaks.
    text = _RE_LINEBREAK_HYPHEN.sub(_dehyphenate, text)

    # 4. Paragraph reflow.
    if reflow:
        text = _reflow(text)

    # 5. Punctuation and whitespace.
    if drop_notes:
        text = _RE_FOOTNOTE_MARKER.sub("", text)
    text = _RE_SPACE_BEFORE_PUNCT.sub(r"\1", text)
    text = _RE_SPACE_AFTER_OPEN.sub(r"\1", text)
    text = _RE_SPACE_BEFORE_CLOSE.sub(r"\1", text)
    text = re.sub(r"[ \t\f\r]+", " ", text)
    text = re.sub(r" *\n *", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()

    if min_alpha_chars and sum(1 for c in text if c.isalpha()) < min_alpha_chars:
        return ""
    return text


if __name__ == "__main__":
    sample = (
        "For use at 10:00 a.m., EDT\n"
        "February 11, 2004\n\n"
        "Board of Governors of the Federal Reserve System\n"
        "Monetary Policy Report to the Congress\n\n"
        "Selected interest rates\nPercent\n2000 2001 2002 2003\n+ 1.2 - 0.5\n"
        "Ten-year Treasury\n\n"
        "Monetary Policy Report \u25aa February 2004\n"
        "The contents of the report indicate that the Committee raised the\n"
        "target for the federal funds rate to 51\u20444 percent, and the communi-\n"
        "Monetary Policy Report \u25aa February 2004\n"
        "cation of policy intentions improved.1 As shown in\n"
        "figure 2, short-\n"
        "and long-term rates fell. On February 14, 2023, the Committee met on\n"
        "a Wednesday.\n\n"
        "NOTE. Data are quarterly and extend through\n"
        "2003:Q4.\n"
        "SOURCE. Department of Commerce.\n"
        "1. See the minutes of the March meeting for\n"
        "further details.\n"
    )
    print(clean_text(sample))

The contents of the report indicate that the Committee raised the target for the federal funds rate to 5 1/4 percent, and the communication of policy intentions improved. As shown in figure 2, short- and long-term rates fell. On February 14, 2023, the Committee met on a Wednesday.


Chunking Phase

This packs the paragrphs into the tiokens, and where the mainsplitting occurs

In [69]:

# Creates a regex pattern to split text at sentence boundaries
# his pattern identifies sentence boundaries by looking for periods, exclamation points, 
# or question marks followed by whitespace. The (?<=) lookbehind ensures we split after the punctuation, keeping the punctuation with the sentence
_SENT = re.compile(r"(?<=[.!?])\s+")

# Splits text into paragraphs based on blank lines.
def _paras(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

# Main chincking functions
def chunk_text(text, target, min_tokens):
    chunks, buf, buf_tok = [], [], 0
    # Takes everything in the buffer and saves it as a chunk, then resets the buffer
    def flush():
        nonlocal buf, buf_tok
        if buf:
            chunks.append("\n\n".join(buf)); buf, buf_tok = [], 0
    for para in _paras(text):
        ptok = n_tokens(para)
        if ptok > target:                          # paragraph too big -> sentences
            flush()
            sbuf, stok = [], 0
            for sent in _SENT.split(para):
                m = n_tokens(sent)
                if m > target:                     # single sentence too big -> hard cut
                    if sbuf:
                        chunks.append(" ".join(sbuf)); sbuf, stok = [], 0
                    ids = enc.encode(sent)
                    for i in range(0, len(ids), target):
                        chunks.append(enc.decode(ids[i:i + target]))
                elif stok + m > target:
                    chunks.append(" ".join(sbuf)); sbuf, stok = [sent], m
                else:
                    sbuf.append(sent); stok += m
            if sbuf:
                chunks.append(" ".join(sbuf))
        elif buf_tok + ptok > target:
            flush(); buf, buf_tok = [para], ptok
        else:
            buf.append(para); buf_tok += ptok
    flush()
    return [c for c in chunks if n_tokens(c) >= min_tokens]

Cleaning Testing Start

Full run 

In [70]:
# These four functions sit downstream of clean_text() - once a scraped FED document has had
# its boilerplate and chart junk stripped out, I still need to chunk it up, throw out anything
# that's too thin to be useful, catch duplicates (a lot of the annual/monetary reports repeat
# whole paragraphs year to year), and tag each chunk with which report it actually came from.
# build_corpus() is the orchestrator that ties all of this together and writes the final jsonl.

def quality_ok(text, min_chars, min_alpha):
    # Final quality gate before a chunk gets written to the corpus. Even after clean_text does
    # its aggressive filtering, a chunk can still slip through that's just a fragment of a table
    # or a stray line of chart data that happened to have enough letters to dodge the alpha-ratio
    # filter earlier, so I check it again here at the chunk level.

    # If the chunk is shorter than min_chars it's not really worth keeping, its probably just a
    # heading or a leftover fragment rather than real body text.
    if len(text) < min_chars:
        return False

    # Same idea as the alpha ratio check in clean_text, count the letters and compare to the
    # total length of the chunk.
    letters = sum(c.isalpha() for c in text)
    # max(len(text), 1) is just there so I don't divide by zero on an empty string, shouldnt
    # happen given the min_chars check above but cheap to guard against.
    return letters / max(len(text), 1) >= min_alpha


def dedup_key(text):
    # A lot of these reports reuse the same boilerplate paragraphs across different years, and
    # sometimes the same file gets picked up twice in the scrape, so I need a cheap way to spot
    # duplicate chunks before they end up in the training corpus twice.

    # Collapsing all whitespace down to single spaces and lowercasing everything first, so two
    # chunks that are identical apart from a line break or capitalization still hash to the same
    # key, otherwise trivial formatting differences would let duplicates through.
    # md5 is just a fingerprint here, not doing anything cryptographic, so its fine that its not
    # collision-resistant against an adversary, I just need something fast and consistent.
    return hashlib.md5(re.sub(r"\s+", " ", text).lower().encode("utf-8")).hexdigest()


def source_of(root, path):
    # I want every chunk tagged with which report family it came from (Monetary Policy Report,
    # Financial Stability Report, etc.) so I can filter or analyze by source later on, and that
    # info is encoded in the folder structure rather than the filename itself.

    # Getting the path relative to whichever input_dir its under, then splitting on the OS
    # separator, the first folder in that relative path is the source category.
    rel = os.path.relpath(path, root).split(os.sep)
    # If the file happens to sit directly in the root with no subfolder, there's no category
    # folder to grab, so I just fall back to the root directory's own name instead.
    return rel[0] if len(rel) > 1 else os.path.basename(os.path.normpath(root))


def build_corpus(input_dirs, out_path, target, min_tokens, min_chars, min_alpha, drop_notes):
    # This is the main driver, walks every input dir for the scraped .txt files, cleans each one,
    # chunks it, filters out the low quality/duplicate chunks, and writes everything that survives
    # out to a single jsonl file that I can actually feed into whatever comes next.

    # Recursively glob every input dir for .txt files, keeping the root alongside each path so
    # source_of() further down knows which root a given file belongs to.
    files = []
    for root in input_dirs:
        files += [(root, p) for p in sorted(glob.glob(os.path.join(root, "**", "*.txt"),
                  recursive=True))]
    # Bailing out early with a helpful message rather than silently writing an empty file, this
    # has saved me a few times when INPUT_DIRS in the Config cell was pointed at the wrong place.
    if not files:
        print("No .txt files found under:", input_dirs, "\n-> check INPUT_DIRS in the Config cell.")
        return 0

    # Making sure the output folder actually exists before I try to open a file inside it.
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    # seen accumulates dedup keys across ALL files, not just within a single file, since the
    # repeated boilerplate problem shows up across different reports, not just within one.
    seen = set(); n_chunks = n_dups = n_low = 0
    with open(out_path, "w", encoding="utf-8") as out:
        for root, path in files:
            # errors="replace" so a stray bad byte in one scraped file doesn't kill the whole run.
            with open(path, encoding="utf-8", errors="replace") as f:
                cleaned = clean_text(f.read(), drop_notes=drop_notes)
            src, kept = source_of(root, path), 0
            for i, chunk in enumerate(chunk_text(cleaned, target, min_tokens)):
                # Skip anything too thin/numeric to be worth keeping.
                if not quality_ok(chunk, min_chars, min_alpha):
                    n_low += 1; continue
                # Skip anything I've already seen, either from this file or an earlier one.
                k = dedup_key(chunk)
                if k in seen:
                    n_dups += 1; continue
                seen.add(k)
                # Writing one json object per line (jsonl) rather than one big json array, so
                # I can stream through this later without loading the whole corpus into memory.
                out.write(json.dumps({
                    "text": chunk, "source": src,
                    "path": os.path.relpath(path), "chunk": i,
                    "n_tokens": n_tokens(chunk),
                }, ensure_ascii=False) + "\n")
                n_chunks += 1; kept += 1
            # Per-file progress line, mainly so I can watch it work through a big batch of reports
            # and sanity check that a given file actually produced chunks.
            print(f"{path}  [{src}]  +{kept}")
    # Summary stats at the end, mostly so I can eyeball whether the dedup/quality thresholds are
    # too aggressive or too lax without having to go dig through the output file myself.
    print(f"\nfiles={len(files)}  chunks={n_chunks}  dups_skipped={n_dups}  "
          f"low_quality_skipped={n_low}\n-> {out_path}")
    return n_chunks

In [71]:
build_corpus(INPUT_DIRS, OUT_PATH, TARGET_TOKENS, MIN_TOKENS,
              MIN_CHARS, MIN_ALPHA, DROP_NOTES)

/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report/1995/1995-ar.txt  [1995]  +129
/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report/1996/1996-ar.txt  [1996]  +125
/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report/1997/1997-ar.txt  [1997]  +124
/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report/1998/1998-ar.txt  [1998]  +142
/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report/1999/1999-ar.txt  [1999]  +132
/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report/2000/2000-ar.txt  [2000]  +125
/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/RESEARCH and SURVAILENCE/annual-report/2001/2001-ar.txt  [2001]  +133
/Users/lorenz

4002

In [72]:
import statistics
from collections import Counter

recs = [json.loads(l) for l in open(OUT_PATH, encoding="utf-8")]
toks = [r["n_tokens"] for r in recs]
print(len(recs), "records")
print("tokens/chunk  min", min(toks), " median", int(statistics.median(toks)), " max", max(toks))
print("by source:", dict(Counter(r["source"] for r in recs)))

4002 records
tokens/chunk  min 158  median 1646  max 2054
by source: {'1995': 129, '1996': 125, '1997': 124, '1998': 142, '1999': 132, '2000': 125, '2001': 133, '2002': 95, '2003': 105, '2004': 102, '2005': 107, '2006': 114, '2007': 132, '2008': 176, '2009': 183, '2010': 231, '2011': 245, '2012': 208, '2013': 216, '2014': 208, '2015': 150, '2016': 156, '2017': 150, '2018': 145, '2019': 126, '2020': 56, '2021': 48, '2022': 48, '2023': 48, '2024': 43}


In [73]:
print("total tokens:", sum(toks))

total tokens: 6438258


In [74]:
print(OUT_PATH)

FED_annualreport_unlablled.jsonl


Token Count

In [78]:
data_dir = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/FED") 

totals = Counter()
grand_total = 0
grand_records = 0

for path in sorted(data_dir.glob("*.jsonl")):
    n_tok = 0
    n_rec = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            n_tok += rec["n_tokens"]
            n_rec += 1
    totals[path.name] = n_tok
    grand_total += n_tok
    grand_records += n_rec
    print(f"{path.name:45s}  {n_rec:7,d} chunks  {n_tok:12,d} tokens")

print("-" * 70)
print(f"{'TOTAL':45s}  {grand_records:7,d} chunks  {grand_total:12,d} tokens")

FED_annualreport_unlablled.jsonl                 4,002 chunks     6,438,258 tokens
FED_speeches_unlablled.jsonl                     5,364 chunks    10,384,997 tokens
FED_testimony_unlablled.jsonl                       92 chunks       128,595 tokens
Financial_Stability_Reports_unlabeled.jsonl        297 chunks       480,600 tokens
fed_Supervison_Regulation_unlabeled.jsonl          151 chunks       230,343 tokens
fed_minutes_unlabeled.jsonl                      1,060 chunks     2,076,029 tokens
fed_monetarypolicyrepos_unlabeled.jsonl          1,138 chunks     1,813,700 tokens
fed_remaining_statements_unlabeled.jsonl           274 chunks       552,138 tokens
fed_unlabeled_beigebooks.jsonl                   6,441 chunks     7,322,027 tokens
----------------------------------------------------------------------
TOTAL                                           18,819 chunks    29,426,687 tokens


In [84]:
data_dir = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/ECB/Token")

# I import the compiled patterns straight from my cleaner so the audit and the cleaner can never drift out of sync, if I add a new
# boiler plate pattern to the cleaner, the audit automatically starts checking for it as well. Because I run this both as a plain
# script and from inside Jupyter (where the working directory is often not the folder the files live in, and where __file__ does
# not even exist when the code is pasted into a cell), the folder containing clean_fed_text.py is located by searching, in order,
# the folder this script lives in, the current working directory, the FED data folder itself, and finally the whole BdF-Project
# tree as a last resort.
def _locate_cleaner():
    searched = []
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parent)
    candidates.append(Path.cwd())
    if data_dir.exists():
        candidates.append(data_dir)
    project_root = data_dir.parent.parent  # .../BdF-Project
    if project_root.exists():
        candidates.append(project_root)
    seen = set()
    for folder in candidates:
        folder = folder.resolve()
        if folder in seen:
            continue
        seen.add(folder)
        searched.append(str(folder))
        if (folder / "clean_fed_text.py").exists():
            sys.path.insert(0, str(folder))
            return
    # The deep search of the project tree, only reached when none of the obvious folders contained the cleaner.
    if project_root.exists():
        for hit in project_root.rglob("clean_fed_text.py"):
            sys.path.insert(0, str(hit.parent))
            return
    sys.exit(
        "clean_fed_text.py was not found, I looked in: %s (and then everywhere under %s). "
        "Move clean_fed_text.py next to this script / the notebook, or run "
        "sys.path.append('<folder containing clean_fed_text.py>') before this."
        % (", ".join(searched), project_root)
    )
 
 
_locate_cleaner()
from clean_fed_text import _CI_LINE, _CS_LINE, _RE_NOTE_START, _RE_SENTENCE_END, _is_junk_line
 
# The JSON field names I expect the text to live under, tried in this order when --text-key is not supplied. If none of these are
# present, I fall back to whichever key holds the longest string value in the record.
_TEXT_KEY_CANDIDATES = ["text", "cleaned_text", "clean_text", "content", "body", "document"]
 
# Classic mojibake sequences ("â€™" for an apostrophe, "Ã©" for é) which mean the document was decoded with the wrong encoding at
# some point and ftfy never got the chance to repair it.
_RE_MOJIBAKE = re.compile("\u00e2\u20ac|\u00c3[\u0080-\u00ff]|\u00c2[\u00a0-\u00bf]")
 
# Unicode artifacts that the normalization part of my cleaner is supposed to have eliminated completely, soft hyphens, zero-width
# characters, the byte order mark, the replacement character, the fraction slash, non-breaking spaces, line/paragraph separators,
# and any C0 control characters other than newline and tab. Finding even one of these means the document never went through the
# normalization.
_RE_UNICODE_LEFTOVER = re.compile(
    "[\u00ad\u200b-\u200f\ufeff\ufffd\u2044\u00a0\u2028\u2029]|[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]"
)
 
# Boiler plate phrases checked as substrings of the whole document, independent of line structure, these should never appear
# anywhere in an adequately cleaned FED document.
_RE_BOILER_PHRASES = re.compile(
    r"for use at \d{1,2}:\d{2}|embargoed for release|letter of transmittal|"
    r"pursuant to section 2b of the federal reserve act|skip to main content|"
    r"return to text|back to top|last update:",
    re.IGNORECASE,
)
 
# A hyphen at the end of a line followed by a lowercase letter means the de-hyphenation never ran on this document,
# "communi-\ncation" should not exist in the cleaned corpus.
_RE_UNJOINED_HYPHEN = re.compile(r"[A-Za-z]{2,}-\n[a-z]")
 
# A mid-line "communi- cation" style leftover, the suspended-hyphen words ("short- and long-term", "one- to three-year") are the
# legitimate exception and are excluded, everything else is suspicious.
_RE_SPLIT_WORD = re.compile(r"[a-z]{3,}- (?!and\b|or\b|to\b|the\b)[a-z]{3,}")
 
# Residual URLs, my drop_notes pass removes the NOTE/SOURCE blocks where these normally live, so a surviving URL usually means a
# note slipped through, it is only a WARN because the body text can very occasionally cite an address legitimately.
_RE_URL = re.compile(r"https?://|www\.", re.IGNORECASE)
 
# The severity of each issue code, FAIL keeps a document out of the training mix, WARN means I should eyeball it.
_SEVERITY = {
    "parse_error": "FAIL",
    "missing_text": "FAIL",
    "empty_or_too_short": "FAIL",
    "mojibake": "FAIL",
    "unicode_leftovers": "FAIL",
    "residual_boilerplate": "FAIL",
    "residual_note_block": "FAIL",
    "duplicate_doc": "FAIL",
    "residual_junk_lines": "WARN",
    "repeated_line": "WARN",
    "unjoined_hyphen": "WARN",
    "split_word_leftover": "WARN",
    "long_glued_token": "WARN",
    "poor_reflow": "WARN",
    "residual_url": "WARN",
    "low_alpha_ratio": "WARN",
}
 
 
def _detect_text_key(record):
    # Tries my candidate names first, then falls back to whichever key holds the longest string, so that differently shaped JSONL
    # files (e.g. from different scraping runs) can still be audited without me remembering each schema.
    for key in _TEXT_KEY_CANDIDATES:
        if isinstance(record.get(key), str):
            return key
    string_keys = [(len(v), k) for k, v in record.items() if isinstance(v, str)]
    if string_keys:
        return max(string_keys)[1]
    return None
 
 
def _check_document(text, doc_id, seen_hashes, args):
    # Runs every per-document check and returns a list of (issue_code, human_readable_detail) tuples, an empty list means PASS.
    issues = []
 
    # Near-empty documents contribute nothing to pre-training and usually indicate that the cleaner stripped everything, which is
    # itself worth knowing about.
    alpha = sum(1 for c in text if c.isalpha())
    if alpha < args.min_alpha:
        issues.append(("empty_or_too_short", "only %d alphabetic characters" % alpha))
        return issues  # The remaining checks are meaningless on an empty document.
 
    # Encoding damage that ftfy should have repaired.
    m = _RE_MOJIBAKE.search(text)
    if m:
        issues.append(("mojibake", "found %r" % m.group(0)))
 
    # Unicode artifacts that the normalization should have eliminated.
    m = _RE_UNICODE_LEFTOVER.search(text)
    if m:
        issues.append(("unicode_leftovers", "found %r" % m.group(0)))
 
    # Exact duplicate detection across the whole corpus (all files together). I hash a whitespace- and case-normalized version so
    # that trivial formatting differences between the HTML and PDF scrape of the same report do not hide the duplication.
    digest = hashlib.sha1(re.sub(r"\s+", " ", text.lower()).encode("utf-8")).hexdigest()
    if digest in seen_hashes:
        issues.append(("duplicate_doc", "identical to %s" % seen_hashes[digest]))
    else:
        seen_hashes[digest] = doc_id
 
    lines = [line.strip() for line in text.split("\n") if line.strip()]
 
    # My own cleaner's patterns re-applied. Any full line that still matches the boiler plate or chart-title lists, or that still
    # looks like a NOTE/SOURCE opener, or that _is_junk_line would have dropped, is direct evidence that the cleaning step failed
    # or was skipped for this document.
    boiler_hits = [line for line in lines if _CI_LINE.fullmatch(line) or _CS_LINE.fullmatch(line)]
    m = _RE_BOILER_PHRASES.search(text)
    if boiler_hits:
        issues.append(("residual_boilerplate", "%d line(s), e.g. %r" % (len(boiler_hits), boiler_hits[0][:80])))
    elif m:
        issues.append(("residual_boilerplate", "phrase %r" % m.group(0)))
    note_hits = [line for line in lines if _RE_NOTE_START.match(line)]
    if note_hits:
        issues.append(("residual_note_block", "%d line(s), e.g. %r" % (len(note_hits), note_hits[0][:80])))
    junk_hits = [line for line in lines if _is_junk_line(line)]
    if junk_hits:
        issues.append(("residual_junk_lines", "%d line(s), e.g. %r" % (len(junk_hits), junk_hits[0][:80])))
 
    # A short line repeated three or more times inside one document is almost certainly a running header that survived both the
    # hardcoded lists and the frequency filter.
    line_counts = Counter(line for line in lines if len(line) >= 10)
    repeated = [(line, n) for line, n in line_counts.items() if n >= 3]
    if repeated:
        line, n = max(repeated, key=lambda item: item[1])
        issues.append(("repeated_line", "%r appears %d times" % (line[:60], n)))
 
    # Now the checks the cleaner cannot see because they are its own blind spots.
    if _RE_UNJOINED_HYPHEN.search(text):
        issues.append(("unjoined_hyphen", "hyphen at line end followed by lowercase, de-hyphenation did not run"))
    m = _RE_SPLIT_WORD.search(text)
    if m:
        issues.append(("split_word_leftover", "found %r" % m.group(0)))
    long_tokens = [tok for tok in text.split() if tok.isalpha() and len(tok) >= 25]
    if long_tokens:
        issues.append(("long_glued_token", "e.g. %r" % long_tokens[0][:40]))
    if _RE_URL.search(text):
        issues.append(("residual_url", "document still contains a URL"))
 
    # Reflow adequacy. After the reflow the bulk of the document should be long paragraph lines, if a large share of the lines are
    # short and do not end like sentences, then the document is still in chopped-up PDF form (headings are also short lines, but a
    # real document only has a handful of them, hence the ratio threshold rather than a per-line rule).
    if len(lines) >= 10:
        short = sum(1 for line in lines if len(line) < 45 and not _RE_SENTENCE_END.search(line))
        ratio = short / len(lines)
        if ratio > 0.4:
            issues.append(("poor_reflow", "%.0f%% of lines are short non-sentences" % (100 * ratio)))
 
    # A sanity check on the overall letter density, FED prose sits around 80%, a document far below that is mostly numbers or
    # symbols and is probably surviving chart debris in bulk.
    if alpha / len(text) < 0.45:
        issues.append(("low_alpha_ratio", "only %.0f%% alphabetic" % (100 * alpha / len(text))))
 
    return issues
 
 
def main(argv=None):
    parser = argparse.ArgumentParser(description="Audit cleaned FED JSONL files before continued pre-training.")
    parser.add_argument("folder", nargs="?", default=str(data_dir),
                        help="folder containing the *.jsonl files (default: %s)" % data_dir)
    parser.add_argument("--text-key", default=None, help="JSON field holding the text (default: auto-detect per file)")
    parser.add_argument("--min-alpha", type=int, default=200, help="minimum alphabetic characters per document (default: 200)")
    parser.add_argument("--max-examples", type=int, default=3, help="examples shown per issue type in the summary (default: 3)")
    parser.add_argument("--max-fail-rate", type=float, default=0.0, help="highest tolerated FAIL fraction before exit code 1 (default: 0.0)")
    parser.add_argument("--report", default=None, help="optionally write one JSON row per WARN/FAIL document to this path")
    args = parser.parse_args(argv)
 
    files = sorted(Path(args.folder).glob("*.jsonl"))
    # If the report is being written into the audited folder itself, it must not be treated as corpus data on this run or on any
    # later run, so the report target is excluded from the audit set.
    if args.report:
        report_path = Path(args.report).resolve()
        files = [path for path in files if path.resolve() != report_path]
    if not files:
        print("No .jsonl files found in %s" % args.folder)
        return 2
 
    # Corpus-wide state, the duplicate hashes deliberately span every file so cross-file duplicates are caught as well.
    seen_hashes = {}
    verdict_counts = Counter()
    issue_doc_counts = Counter()
    issue_examples = defaultdict(list)
    report_rows = []
    total_docs = total_words = total_chars = 0
 
    print("Auditing %d JSONL file(s) in %s\n" % (len(files), Path(args.folder).resolve()))
    for path in files:
        file_verdicts = Counter()
        text_key = args.text_key  # Auto-detection is done per file, since different scraping runs used different field names.
        with open(path, encoding="utf-8") as handle:
            for lineno, raw in enumerate(handle, 1):
                raw = raw.strip()
                if not raw:
                    continue
                doc_id = "%s:%d" % (path.name, lineno)
                issues = []
                text = None
                # A JSONL line can fail to parse, parse to a bare string, or parse to an object whose text field I have to find.
                try:
                    record = json.loads(raw)
                except json.JSONDecodeError as exc:
                    issues.append(("parse_error", str(exc)[:80]))
                    record = None
                if record is not None:
                    if isinstance(record, str):
                        text = record
                    elif isinstance(record, dict):
                        if text_key is None:
                            text_key = _detect_text_key(record)
                        if text_key and isinstance(record.get(text_key), str):
                            text = record[text_key]
                        elif args.text_key is None:
                            # Mixed schemas inside one file do happen across my scraping runs, so when I am auto-detecting (i.e.
                            # --text-key was not forced), a record whose text lives under a different key is re-detected on its
                            # own rather than being wrongly reported as missing.
                            fallback_key = _detect_text_key(record)
                            if fallback_key and isinstance(record.get(fallback_key), str):
                                text = record[fallback_key]
                    if text is None and not issues:
                        issues.append(("missing_text", "no string text field found (looked for %s)" % ", ".join(_TEXT_KEY_CANDIDATES)))
                if text is not None:
                    issues.extend(_check_document(text, doc_id, seen_hashes, args))
                    total_words += len(text.split())
                    total_chars += len(text)
 
                # The document verdict is the worst severity among its issues.
                severities = {_SEVERITY[code] for code, _ in issues}
                verdict = "FAIL" if "FAIL" in severities else ("WARN" if severities else "PASS")
                total_docs += 1
                verdict_counts[verdict] += 1
                file_verdicts[verdict] += 1
                for code, detail in issues:
                    issue_doc_counts[code] += 1
                    if len(issue_examples[code]) < args.max_examples:
                        issue_examples[code].append("%s -> %s" % (doc_id, detail))
                if issues:
                    report_rows.append({"doc": doc_id, "verdict": verdict,
                                        "issues": [{"code": code, "detail": detail} for code, detail in issues]})
        print("  %-40s %5d docs | %5d pass | %4d warn | %4d fail"
              % (path.name, sum(file_verdicts.values()), file_verdicts["PASS"], file_verdicts["WARN"], file_verdicts["FAIL"]))
 
    if total_docs == 0:
        print("\nThe JSONL files contained no documents.")
        return 2
 
    # The corpus summary, this is what I read before deciding whether the corpus is ready for the training run.
    fail_rate = verdict_counts["FAIL"] / total_docs
    print("\nCorpus summary")
    print("  documents ............. %d" % total_docs)
    print("  words / characters .... %d / %d" % (total_words, total_chars))
    print("  pass / warn / fail .... %d / %d / %d  (%.1f%% fail)"
          % (verdict_counts["PASS"], verdict_counts["WARN"], verdict_counts["FAIL"], 100 * fail_rate))
 
    if issue_doc_counts:
        print("\nIssues by number of affected documents")
        for code, count in issue_doc_counts.most_common():
            print("  [%s] %-22s %5d doc(s)" % (_SEVERITY[code], code, count))
            for example in issue_examples[code]:
                print("        e.g. %s" % example)
 
    if args.report:
        # One JSON row per flagged document, so I can grep the report, join it back onto the corpus, or feed the FAIL ids into a
        # filtering step that excludes them from the training mix.
        with open(args.report, "w", encoding="utf-8") as handle:
            for row in report_rows:
                handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        print("\nWrote %d flagged document(s) to %s" % (len(report_rows), args.report))
 
    # The gate itself, strict by default, any FAIL document makes the corpus inadequate unless I explicitly raise --max-fail-rate.
    if fail_rate > args.max_fail_rate:
        print("\nVerdict: NOT adequate for pre-training (fail rate %.2f%% exceeds the allowed %.2f%%)."
              % (100 * fail_rate, 100 * args.max_fail_rate))
        return 1
    print("\nVerdict: adequate for pre-training%s."
          % (" (with %d document(s) to eyeball)" % verdict_counts["WARN"] if verdict_counts["WARN"] else ""))
    return 0
 
 
if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        # Inside a Jupyter notebook the command line belongs to the kernel (it passes things like "-f kernel.json" which would
        # confuse argparse), so the arguments are ignored and the defaults (data_dir) are used. sys.exit is also avoided here
        # because in a notebook it only produces the unhelpful "An exception has occurred, use %tb" message. To audit a different
        # folder from a notebook, call main(["path/to/folder"]) directly.
        main([])
    else:
        # The BrokenPipeError guard keeps the script well behaved when I pipe its output into head or grep.
        try:
            sys.exit(main())
        except BrokenPipeError:
            sys.exit(0)

Auditing 14 JSONL file(s) in /Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/ECB/Token

  ecb_Economic_Bulletin_unlabeled.jsonl      803 docs |   753 pass |   50 warn |    0 fail
  ecb_Financial_Stability_reports_unlabeled.jsonl    64 docs |    51 pass |   13 warn |    0 fail
  ecb_annual_report_archive_unlabeled.jsonl   860 docs |   726 pass |  134 warn |    0 fail
  ecb_bsu_speeches_unlabeled.jsonl           615 docs |   605 pass |    9 warn |    1 fail
  ecb_fsr_archive_unlabeled.jsonl            698 docs |   636 pass |   59 warn |    3 fail
  ecb_macroprudential_bulletin_unlabeled.jsonl   108 docs |   104 pass |    4 warn |    0 fail
  ecb_mep_letters_unlabeled.jsonl            227 docs |   169 pass |   58 warn |    0 fail
  ecb_monetary_policy_accounts_unlabeled.jsonl   477 docs |   477 pass |    0 warn |    0 fail
  ecb_monthly_bulletin_unlabeled.jsonl      5418 docs |  3998 pass | 1413 warn |    7 fail
  ecb_other_gc_decisions_unlabeled.jsonl